# Data exploration

Load the processed aerodynamic dataset from Google Drive. The raw archive is not needed for routine exploration or modeling.

The raw-to-processed conversion remains in `src.data.loaders` for reproducibility, but only needs to be run again if the processed CSV must be regenerated.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive
drive.mount('/content/drive')

# Keep the processed dataset in Drive; keep the source code in /content.
DRIVE_DATA = Path('/content/drive/MyDrive/airfoil-ml-surrogate/data')
PROCESSED_PATH = DRIVE_DATA / 'processed' / 'airfoil_learning.csv'
PROJECT_ROOT = Path('/content/airfoil-ml-surrogate')

if not PROCESSED_PATH.exists():
    raise FileNotFoundError(f'Processed dataset not found: {PROCESSED_PATH}')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'Source repository not found: {PROJECT_ROOT}. '
        'Clone or upload the repository into /content first.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data.preprocessing import fit_transform_features, geometry_aware_split
from src.data.validation import validate_dataframe

df = pd.read_csv(PROCESSED_PATH)
validate_dataframe(df)
print(f'Loaded: {PROCESSED_PATH}')
print(f'Shape: {df.shape}')
df.head()

## Leakage-safe partitions

The split keeps every operating condition for a geometry in one partition. The scaler is fitted on training features only.

In [ ]:
geometry_columns = sorted(column for column in df.columns if column.startswith('geom_'))
feature_columns = geometry_columns + ['Alpha', 'Re']
if 'Ncrit' in df.columns:
    feature_columns.append('Ncrit')
target_columns = ['Cl', 'Cd']

split = geometry_aware_split(df, df['geometry_id'])
X_train, X_validation, X_test, scaler = fit_transform_features(
    split.train[feature_columns].to_numpy(),
    split.validation[feature_columns].to_numpy(),
    split.test[feature_columns].to_numpy(),
)

print(f'Train rows: {len(split.train):,}')
print(f'Validation rows: {len(split.validation):,}')
print(f'Test rows: {len(split.test):,}')
print(f'Features: {len(feature_columns)}')
print(f'Unique geometries: {df.geometry_id.nunique():,}')

In [ ]:
import matplotlib.pyplot as plt

display(df.describe(include='all').T)
df[target_columns].hist(figsize=(10, 4))
plt.tight_layout()
plt.show()